# Lecture 8 — Kaggle competition 1: bike rentals per hour

The course functions in this notebook are adapted from the Machine Learning Specialization labs by Andrew Ng (DeepLearning.AI and Stanford Online). The data is the UCI Bike Sharing dataset by Hadi Fanaee-T, published under the CC BY 4.0 licence. Its authors ask users to cite Fanaee-T and Gama (2013).

This notebook is the starting point for the first Kaggle competition of the course. You predict how many bikes are rented in each hour of the test days. The notebook loads the data and fits a first linear regression with gradient descent. Section 4 writes a file that you can submit. Then it is your turn to improve the model.

Before you start, save your own copy with File > Save a copy in Drive.

The notebook needs the three functions you wrote in Exercise 5 of `MI-L08-vectorization`. Paste them into the first code cell of Section 3. Then run all the cells with Runtime > Run all in Colab. If the notebook cannot load the data, Section 1.2 tells you how to upload it.

If a check of Exercise 5 does not pass yet, finish Exercise 5 first. Until then you can submit `Sample_Solution.csv` from the Data tab of the competition, to see how a submission works.

## Competition rules

- You join the competition with the invitation link in the Lecture 8 block on Moodle.
- You predict `count`, the number of bikes rented in each hour of `Test.csv`.
- Submissions are scored with the root mean squared error (RMSE) between your predictions and the true counts. Lower is better. Section 3.1 explains the RMSE.
- `Sample_Solution.csv` is a valid submission. It predicts 192, the mean count in `Train.csv`, for every hour. It scores about 181 on the public leaderboard. That is the first number to beat.
- The competition closes on Tuesday 10 November 2026 at 12:00 CET, which is 11:00 UTC. That is 30 minutes before Lecture 9 starts.
- You can work alone or in a team of up to 3 students. Each student can be in one team only.
- You can submit up to 5 files a day. Kaggle starts a new day at 00:00 UTC, which is 01:00 in Denmark. Before the deadline, select up to 2 of your submissions for the final ranking.
- You can still submit after the deadline. Kaggle scores these late submissions, but they do not count in the final ranking.
- Use only the files of this competition. External data is not allowed, in particular any copy of the UCI Bike Sharing dataset, because it contains the answers.
- The competition is not graded. Your position on the leaderboard does not count towards the exam or any mark.
- The public leaderboard is not the final score. During the competition it shows your score on half of the test days. The final ranking uses the other half, which nobody sees until the competition closes.

# 1 Setup and data

## 1.1 Tools
- NumPy for the computations
- pandas to read and write the CSV files
- Matplotlib for the plots

In [ ]:
import copy, glob, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)  # print NumPy arrays with two decimals

## 1.2 Data files

The competition has two data files. `Train.csv` has the hours of days 1 to 19 of each month, with the answer in the column `count`. `Test.csv` has the hours from the 20th to the end of each month, without the answer.

The notebook downloads both files from the folder `lecture08/data` of the course repository on GitHub. They are the same files as on the Data tab of the competition. The first two lines of the output say where each file came from.

If the download fails, upload the two files yourself. Download `Train.csv` and `Test.csv` from the Data tab of the competition on Kaggle. Then open the Files panel on the left of Colab and upload both files. The notebook finds them there when the download fails. If you run the notebook on your own computer, put the two files in the folder of the notebook. Colab deletes uploaded files when the runtime is disconnected, for example after a long time without use. If that happens, upload the files again.

In [ ]:
# The two competition files in the course repository on GitHub
TRAIN_URL = "https://raw.githubusercontent.com/andresmasegosa/MI_E26/main/lecture08/data/Train.csv"
TEST_URL = "https://raw.githubusercontent.com/andresmasegosa/MI_E26/main/lecture08/data/Test.csv"

In [ ]:
def load_competition_file(name, url):
    """
    Loads one competition file, Train.csv or Test.csv, as a pandas DataFrame.
    It tries the copy in the course repository on GitHub first. Then it looks for
    the file next to this notebook, in /content, in a folder called data and in the
    input folder of a Kaggle notebook. It prints where the file came from.

    Args:
      name (str): file name, "Train.csv" or "Test.csv"
      url (str) : address of the file on GitHub

    Returns:
      df (DataFrame): the rows of the file
    """
    try:
        df = pd.read_csv(url)
        print(f"Loaded {len(df)} rows of {name} from GitHub")
        return df
    except Exception:
        pass

    # the download failed, so look for a local copy
    paths = [name, os.path.join("/content", name), os.path.join("data", name)]
    # glob finds the file in any folder under /kaggle/input
    paths += sorted(glob.glob(os.path.join("/kaggle/input", "**", name), recursive=True))
    for path in paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {len(df)} rows of {name} from the file {path}")
            return df
    raise FileNotFoundError(
        f"Could not load {name} from GitHub or from a local copy. Check your internet "
        "connection, or download Train.csv and Test.csv from the Data tab of the "
        "competition on Kaggle. In Colab, open the Files panel on the left and upload "
        "both files. Then run this cell again.")

In [ ]:
# load the two files and check their columns
train = load_competition_file("Train.csv", TRAIN_URL)
test = load_competition_file("Test.csv", TEST_URL)

columns = ["Id", "date", "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
           "weathersit", "temp", "atemp", "hum", "windspeed"]
assert list(train.columns) == columns + ["count"], \
    "Train.csv does not have the expected columns. Check that you downloaded the files of this competition."
assert list(test.columns) == columns, \
    "Test.csv does not have the expected columns. Check that you downloaded the files of this competition."
# untouched copies, so that the check of Section 6 can see if train or test was changed
train_loaded, test_loaded = train.copy(), test.copy()

print(f"Train: {len(train)} hours on {train['date'].nunique()} days")
print(f"Test:  {len(test)} hours on {test['date'].nunique()} days")
train.head()

The output first says where each file was loaded from. Then come `Train: 10886 hours on 456 days`, `Test:  6493 hours on 275 days` and the first five rows of the training data.

## 1.3 The columns

Each row is one hour of 2011 or 2012 in Washington, D.C. A few hours are missing in the source data, so some days have fewer than 24 rows.

| Column | Meaning |
|---|---|
| Id | A number that identifies the hour. It says nothing about the time. |
| date | The day, as yyyy-mm-dd. We use it to make a validation set from whole days. |
| season | 1 winter, 2 spring, 3 summer, 4 autumn. Season 1 runs from 21 December to 20 March. |
| yr | 0 for 2011 and 1 for 2012 |
| mnth | Month, 1 to 12 |
| hr | Hour of the day, 0 to 23 |
| holiday | 1 if the day is a public holiday in Washington, D.C. |
| weekday | Day of the week, 0 for Sunday to 6 for Saturday |
| workingday | 1 if the day is neither a weekend day nor a holiday |
| weathersit | 1 clear or partly cloudy, 2 mist or clouds, 3 light rain or light snow, 4 heavy rain or snow |
| temp | Temperature scaled to the range 0 to 1, where 0 is -8 °C and 1 is 39 °C |
| atemp | Feels-like temperature scaled to the range 0 to 1, where 0 is -16 °C and 1 is 50 °C |
| hum | Relative humidity divided by 100 |
| windspeed | Wind speed, scaled by the dataset's authors (divided by 67) |
| count | Number of bikes rented in that hour, only in Train.csv |

# 2 Look at the data

The plot shows the average number of rentals in each hour of the day. There is one line for working days and one for weekends and holidays.

In [ ]:
# average count in each hour of the day, for working days and for the other days
hourly = train.groupby(["workingday", "hr"])["count"].mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hourly.loc[1].index, hourly.loc[1].values, marker="o", label="working day")
ax.plot(hourly.loc[0].index, hourly.loc[0].values, marker="o", label="weekend or holiday")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour of the day (hr)")
ax.set_ylabel("average count")
ax.set_title("Average number of rentals in each hour, Train.csv")
ax.legend()
plt.show()

### Question 1

Answer these questions in the cell below.

1. How does the number of rentals change over a working day? And over a weekend or a holiday?
2. The first model in Section 4 uses `hr` as one number with one weight. Keep the other features fixed. What does its prediction look like as a function of the hour? Can it follow these curves?

Write your answers here.

1. ...
2. ...

# 3 The course functions

The model uses the three functions that you wrote in Exercise 5 of the notebook `MI-L08-vectorization`. `compute_model_output` computes $\mathbf{X}\mathbf{w} + b$ for all the rows at once. `compute_cost` is the cost

$$J(\mathbf{w},b) = \frac{1}{2m} \sum_{i=1}^{m} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

and `compute_gradient` returns its partial derivatives, `dj_db` first and then `dj_dw`. The Lecture 7 notebooks returned them in the other order.

Copy your three functions from `MI-L08-vectorization` and paste them into the next cell, in place of the three functions that are there. Keep their names and their arguments. You can also fill in the lines marked `# TODO` again. Do not take the functions of the Exercise 1-4 notebook. They loop over the houses, and on 10886 hours each fit would take minutes instead of a second.

The cell after it checks your functions on a small example and prints `Check passed`. If a function is wrong or too slow, it stops with a message that says what to fix. After the check comes `gradient_descent`, the same function as in the lecture.

In [ ]:
# Paste your compute_model_output, compute_cost and compute_gradient from Exercise 5 of
# MI-L08-vectorization into this cell, in place of the three functions below.
def compute_model_output(X, w, b):
    """
    Predictions of the linear model for all the examples
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      f_wb (ndarray (m,)): prediction for every example
    """
    f_wb = None  # TODO: the predictions for all the rows of X, without a loop

    return f_wb


def compute_cost(X, y, w, b):
    """
    compute cost, vectorized
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    err = None   # TODO: the error vector e, shape (m,)
    cost = None  # TODO: the cost J, a scalar

    return cost


def compute_gradient(X, y, w, b):
    """
    Computes the gradient for linear regression, vectorized
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      dj_db (scalar):       The partial derivative of the cost with respect to b
      dj_dw (ndarray (n,)): The partial derivatives of the cost with respect to the weights in w
    """
    m = X.shape[0]
    err = None    # TODO: the error vector e, shape (m,)
    dj_db = None  # TODO: the partial derivative with respect to b, a scalar
    dj_dw = None  # TODO: the partial derivatives with respect to w, shape (n,)

    return dj_db, dj_dw

In [ ]:
# Check your result
# a small example whose results are known
X_chk = np.array([[1.0, 2.0], [3.0, -1.0], [0.5, 4.0]])
y_chk = np.array([3.0, 1.0, 2.0])
w_chk = np.array([0.5, -1.0])
b_chk = 2.0

f_chk = compute_model_output(X_chk, w_chk, b_chk)
assert f_chk is not None, "compute_model_output returns None. In the cell above, replace the compute_model_output that returns None with yours from Exercise 5a, so that only yours is left. Then run that cell and this one again."
assert np.shape(f_chk) == (3,), f"compute_model_output should return shape (3,), not {np.shape(f_chk)}"
assert np.allclose(f_chk, [0.5, 4.5, -1.75]), "compute_model_output gives wrong predictions. Paste the compute_model_output that passed the check of Exercise 5a."

cost_chk = compute_cost(X_chk, y_chk, w_chk, b_chk)
assert cost_chk is not None, "compute_cost returns None. In the cell above, replace the compute_cost that returns None with yours from Exercise 5b, so that only yours is left. Then run that cell and this one again."
assert np.ndim(cost_chk) == 0, f"compute_cost returns an array of shape {np.shape(cost_chk)}, but J(w,b) is one number."
assert np.isclose(cost_chk, 5.4270833), "compute_cost gives a wrong cost. Paste the compute_cost that passed the check of Exercise 5b."

dj_db_chk, dj_dw_chk = compute_gradient(X_chk, y_chk, w_chk, b_chk)
assert dj_db_chk is not None and dj_dw_chk is not None, "compute_gradient returns None. In the cell above, replace the compute_gradient that returns None with yours from Exercise 5c, so that only yours is left. Then run that cell and this one again."
assert np.ndim(dj_db_chk) == 0, "The first value that compute_gradient returns should be dj_db, a scalar. Return dj_db first and then dj_dw, as in MI-L08-vectorization."
assert np.shape(dj_dw_chk) == (2,), f"dj_dw should have shape (2,) here, not {np.shape(dj_dw_chk)}"
assert np.isclose(dj_db_chk, -0.9166667) and np.allclose(dj_dw_chk, [2.0416667, -7.8333333]), "compute_gradient gives wrong partial derivatives. Paste the compute_gradient that passed the check of Exercise 5c."

# gradient descent calls compute_cost and compute_gradient once per iteration, so they must be fast
X_big = np.random.default_rng(0).normal(size=(10000, 12))
y_big = np.zeros(10000)
t_cost = %timeit -o -q -n 5 -r 3 compute_cost(X_big, y_big, np.zeros(12), 0.)
t_grad = %timeit -o -q -n 5 -r 3 compute_gradient(X_big, y_big, np.zeros(12), 0.)
ms_cost, ms_grad = 1000 * t_cost.best, 1000 * t_grad.best
assert ms_cost < 2 and ms_grad < 2, f"On 10000 rows compute_cost takes {ms_cost:0.2f} ms and compute_gradient {ms_grad:0.2f} ms. A fit calls each of them 10000 times, so it would take about {10 * (ms_cost + ms_grad):0.0f} seconds. Paste the versions without loops from Exercise 5."
print("Check passed")

The expected output is `Check passed`.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking
    num_iters gradient steps with learning rate alpha

    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent

    Returns:
      w (ndarray (n,))    : Updated values of parameters
      b (scalar)          : Updated value of parameter
      J_history (list)    : cost after each iteration
      """

    # A list to store the cost J at each iteration, for the plots later
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):

        # Calculate the gradient at the current w and b
        dj_db, dj_dw = gradient_function(X, y, w, b)

        # Update the parameters using alpha and the gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            J_history.append( cost_function(X, y, w, b))

        # Print the cost 10 times during the run, or at every iteration if num_iters < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")

    return w, b, J_history

`zscore_normalize_features` has one change from the lecture. A column with the same value in every row has standard deviation 0. Dividing by 0 gives `nan`. The function sets that standard deviation to 1. The column then becomes 0 in every row. Gradient descent leaves its weight at 0.

In [ ]:
def zscore_normalize_features(X):
    """
    Scales each column of X with z-score normalization

    Args:
      X (ndarray (m,n))     : input data, m examples, n features

    Returns:
      X_norm (ndarray (m,n)): X with every column scaled
      X_mean (ndarray (n,)) : mean of each feature
      X_sigma (ndarray (n,)): standard deviation of each feature
    """
    # find the mean of each column/feature
    X_mean     = np.mean(X, axis=0)                 # X_mean has shape (n,)
    # find the standard deviation of each column/feature
    X_sigma  = np.std(X, axis=0)                  # X_sigma has shape (n,)
    # a constant column has standard deviation 0. Use 1 so that the division is defined
    X_sigma[X_sigma == 0] = 1
    # NumPy subtracts X_mean from every row of X and divides every row by X_sigma. This is called broadcasting.
    X_norm = (X - X_mean) / X_sigma

    return (X_norm, X_mean, X_sigma)

## 3.1 The metric

The competition is scored with the root mean squared error

$$\text{RMSE} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}\left(\hat{y}^{(i)} - y^{(i)}\right)^2}$$

where $m$ is the number of hours, $\hat{y}^{(i)}$ is the prediction for hour $i$ and $y^{(i)}$ is the true count. The RMSE is measured in bikes, so it is the size of a typical error. The cost $J$ is half the mean squared error, so $\text{RMSE} = \sqrt{2J}$ when both are computed on the same rows. Gradient descent makes $J$ small on the training rows. That is the same as making the RMSE small on those rows.

In [ ]:
def rmse(predictions, y):
    """
    Root mean squared error
    Args:
      predictions (ndarray (m,)): predicted values
      y (ndarray (m,))          : true values

    Returns:
      rmse (scalar): square root of the mean of the squared errors
    """
    return np.sqrt(np.mean((predictions - y)**2))

# 4 A first model and a first submission

## 4.1 A validation set of whole days

To compare models we need hours whose count the model has not seen. The test set is made of whole days that the model never sees, the 20th to the end of each month. We make the validation set the same way. Days 15 to 19 of each month are the validation set, and the model is fitted on days 1 to 14.

Hours of the same day are alike, because they share the weather and the kind of day. A validation set of single hours would mix them with training hours of the same day. It would then tell us less about days the model has never seen.

In [ ]:
# days 15 to 19 of each month are the validation set
day_of_month = train["date"].str[8:10].astype(int).to_numpy()
is_val = day_of_month >= 15

y_all = train["count"].to_numpy(dtype=float)
y_fit, y_val = y_all[~is_val], y_all[is_val]
print(f"Fitting days: {np.sum(~is_val)} hours. Validation days: {np.sum(is_val)} hours.")

## 4.2 Features

`build_features` takes a table with the columns of `Train.csv` or `Test.csv` and returns the matrix $\mathbf{X}$ and the names of its columns. It selects the columns by name. We use the same function for Train and for Test, so the columns of the two matrices always match. The first model uses the 12 columns from `season` to `windspeed` as numbers.

In [ ]:
def build_features(df):
    """
    Features of the first model. They are the 12 columns season ... windspeed, as numbers
    Args:
      df (DataFrame): rows of Train.csv or Test.csv

    Returns:
      X (ndarray (m,12)): one row per hour
      names (list)      : the name of each column of X
    """
    names = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
             "weathersit", "temp", "atemp", "hum", "windspeed"]
    X = df[names].to_numpy(dtype=float)
    return X, names

## 4.3 Fitting and validation

`fit_linear_regression` z-scores the features with the mean and the standard deviation of the rows it is given. Then it runs gradient descent from $\mathbf{w} = \mathbf{0}$, $b = 0$. It also returns that mean and standard deviation, because any other rows must be scaled with these same values before we predict them. This is the rule of Section 6.2 of the notebook `MI-L08-multiple-linear-regression`.

We fit the first model on days 1 to 14 with $\alpha = 0.1$ and 10000 iterations, and measure it on days 15 to 19. To have a reference, the last line gives the validation RMSE of a model that predicts the mean count of the fitting days for every hour.

In [ ]:
def fit_linear_regression(X, y, alpha, num_iters):
    """
    Z-scores X with its own mean and standard deviation, then runs gradient descent
    from w = 0, b = 0
    Args:
      X (ndarray (m,n)): training data, m examples with n features
      y (ndarray (m,)) : target values
      alpha (float)    : learning rate
      num_iters (int)  : number of iterations of gradient descent

    Returns:
      w (ndarray (n,))    : parameters found by gradient descent
      b (scalar)          : parameter found by gradient descent
      mu (ndarray (n,))   : mean of each feature, used for the z-score
      sigma (ndarray (n,)): standard deviation of each feature, used for the z-score
      J_history (list)    : cost after each iteration
    """
    X_norm, mu, sigma = zscore_normalize_features(X)
    w, b, J_history = gradient_descent(X_norm, y, np.zeros(X.shape[1]), 0.,
                                       compute_cost, compute_gradient, alpha, num_iters)
    return w, b, mu, sigma, J_history

In [ ]:
X_all, feature_names = build_features(train)
X_fit, X_val = X_all[~is_val], X_all[is_val]

# fit on days 1 to 14
w, b, mu, sigma, J_hist = fit_linear_regression(X_fit, y_fit, alpha=0.1, num_iters=10000)
print(f"\nCost J on the fitting days: {J_hist[-1]:0.2f}, so sqrt(2J) = {np.sqrt(2 * J_hist[-1]):0.2f}")

# predict the validation days, scaled with the mean and standard deviation of the fitting days
p_val = compute_model_output((X_val - mu) / sigma, w, b)
print(f"Negative predictions on the validation days: {np.sum(p_val < 0)} of {len(p_val)}")
p_val = np.clip(p_val, 0, None)
rmse_val_baseline = rmse(p_val, y_val)
print(f"Validation RMSE of the first model: {rmse_val_baseline:0.2f}")
print(f"Validation RMSE when we predict the mean count of the fitting days: "
      f"{rmse(np.full(len(y_val), np.mean(y_fit)), y_val):0.2f}")

The expected output ends with these lines.

```
Cost J on the fitting days: 9849.25, so sqrt(2J) = 140.35
Negative predictions on the validation days: 83 of 2860
Validation RMSE of the first model: 145.68
Validation RMSE when we predict the mean count of the fitting days: 183.59
```

The first model beats the mean. Its typical error is still about 146 bikes, while the mean count of an hour is 192.

On some hours the first model predicts a negative number of bikes. We clip every prediction at 0 before we measure it or submit it. Question 4 in Section 8 comes back to this.

## 4.4 Retrain on all of Train and write a submission

We used the validation days to measure the model. For the submission we fit the same model again on all the rows of Train, days 1 to 19, because more data gives better estimates of the weights. Then we predict the test hours, clip at 0 and write the file.

In [ ]:
def write_submission(ids, predictions, filename="submission.csv"):
    """
    Writes a submission file with the columns Id and Expected, one row per test hour
    Args:
      ids (Series or ndarray (m,)): the Id of each test hour, in the order of Test.csv
      predictions (ndarray (m,))  : predicted count for each test hour
      filename (str)              : name of the file to write
    """
    predictions = np.asarray(predictions, dtype=float)
    assert len(predictions) == len(ids), f"There are {len(ids)} test hours but {len(predictions)} predictions."
    assert np.all(np.isfinite(predictions)), "Some predictions are nan or inf. Look at your features and the learning rate."
    assert np.all(predictions >= 0), "Some predictions are negative. Clip them at 0 first."
    submission = pd.DataFrame({"Id": np.asarray(ids), "Expected": np.round(predictions, 2)})
    submission.to_csv(filename, index=False)
    print(f"Wrote {filename} with {len(submission)} rows and the columns Id and Expected")

In [ ]:
# fit the first model on all of Train, days 1 to 19
w, b, mu, sigma, _ = fit_linear_regression(X_all, y_all, alpha=0.1, num_iters=10000)

# predict the test hours with the mean and standard deviation of Train, and clip at 0
X_test, _ = build_features(test)
p_test = np.clip(compute_model_output((X_test - mu) / sigma, w, b), 0, None)
write_submission(test["Id"], p_test)

The expected output ends with `Wrote submission.csv with 6493 rows and the columns Id and Expected`.

To submit, open the Files panel on the left of Colab, download `submission.csv` and upload it on the competition page on Kaggle. This first file scores about 144 on the public leaderboard.

# 5 Ideas to try

The first model is a long way from the best that linear regression can do here. Below are four ideas. Try them one at a time in Section 6, and keep the ones that lower the validation RMSE. If you change one thing at a time, you know which change helped.

- Replace `hr` by 24 columns, one for each hour of the day. The column of hour $k$ is 1 in the rows of that hour and 0 in all other rows. This is called one-hot encoding. Each hour then gets its own weight, so the model can follow the peaks of the plot in Section 2 instead of a straight line.
- Add the products of these 24 hour columns with `workingday`. Working days and the other days have very different curves in Section 2. With these columns each kind of day gets its own weight for every hour.
- Add powers of the temperature, such as $\text{temp}^2$ and $\text{temp}^3$. People ride more when it is warm, but the effect may level off on the hottest days. A straight line in `temp` cannot show that. Centre the temperature before you take the powers, for example with `temp - 0.5`. The powers are then less alike, and gradient descent converges faster.
- Let the model learn $\log(1 + \text{count})$ instead of the count, and turn each of its predictions $p$ back into a count with $e^{p} - 1$. In Section 6 you do this in `target_from_count` and `count_from_target`. The reason is that the weather changes the count by a factor. On working days in `Train.csv`, an hour of light rain or snow has between half and two thirds of the rentals of a clear hour at the same time of day. That is about 210 bikes fewer at 17 and 2 fewer at 3 in the morning. A linear model adds the same number of bikes at every hour. After the logarithm, a factor becomes a fixed number that the model can add.

Use the validation RMSE to decide, not the public leaderboard.

# 6 Your model

The cell below holds everything you change.

- `build_my_features` builds your features from a table with the columns of `Train.csv` or `Test.csv`. Use only columns that `Test.csv` has as well. Build every new column inside this function, for example `t = df["temp"].to_numpy() - 0.5`, and join the columns with `np.column_stack`. The notebook gives the function a copy of each table, so `train` and `test` stay as they were loaded. Do not change `train`, `test`, `y_all`, `y_fit` or `y_val` anywhere else. The check cell stops if they have changed.
- To centre a column, subtract a fixed number, such as the 0.5 above. Do not use the mean of `df` or another statistic of the whole table, and do not call `zscore_normalize_features` here. The test rows would then be centred with their own values, not with those of the training rows. `fit_linear_regression` already scales every column with the mean and the standard deviation of the training rows.
- Row i of the matrix that `build_my_features` returns must be the hour in row i of the table it gets. If you sort the rows, put them back in their original order before you return `X`.
- `target_from_count` turns the count into what the model learns, and `count_from_target` turns the model's predictions back into counts.
- `alpha` and `num_iters` are the settings of gradient descent.

As they are, they give the first model of Section 4. Change one thing at a time. After each change, run this cell and the next two. Then compare your validation RMSE with the first model's.

If Colab has restarted the runtime, first select this cell and use Runtime > Run before. That runs every cell above this one.

In [ ]:
# Your model: change the code between START and END
### START CODE HERE ###
def build_my_features(df):
    """
    Your features, built from the columns of Train.csv or Test.csv
    Args:
      df (DataFrame): a copy of the rows of Train.csv or Test.csv

    Returns:
      X (ndarray (m,n)): one row per hour, in the order of the rows of df
      names (list)     : the name of each column of X
    """
    X, names = build_features(df)
    return X, names


def target_from_count(count):
    """What the model learns. The first model learns the count itself."""
    return count


def count_from_target(target):
    """Turns predictions of the target back into counts."""
    return target


alpha = 0.1
num_iters = 10000
### END CODE HERE ###

In [ ]:
# Check your model
# build_and_check_my_features builds your features and checks them. The next cell and the cell of
# Section 7 call it again, so they always use the cell of your model as you last ran it.
def build_and_check_my_features():
    """
    Builds your features for all of Train and for Test with build_my_features, and checks them.
    First it checks that train, test, y_all, y_fit and y_val have not changed since they were made.
    build_my_features gets a copy of each table, so what it writes into df stays in the copy.

    Returns:
      X_all_my (ndarray (m,n)) : your features for every row of Train
      X_test_my (ndarray (m,n)): your features for every row of Test
    """
    assert train.equals(train_loaded) and test.equals(test_loaded), \
        "train or test has changed since Section 1.2. Build new columns inside build_my_features, not in train or test. Remove the line that changes them, then use Runtime > Run all."
    assert np.array_equal(y_all, train["count"].to_numpy(dtype=float)) and np.array_equal(y_fit, y_all[~is_val]) and np.array_equal(y_val, y_all[is_val]), \
        "y_all, y_fit or y_val has changed since Section 4.1. Change the target in target_from_count, not in these arrays. Remove the line that changes them, then use Runtime > Run all."
    X_all_my, my_names = build_my_features(train.copy())
    try:
        X_test_my, _ = build_my_features(test.copy())
    except KeyError as e:
        missing = [c for c in train.columns if c not in test.columns and repr(c) in str(e)] or [str(e)]
        raise AssertionError(f"build_my_features works on Train but not on Test. It uses the column {', '.join(missing)}, "
                             "which Test.csv does not have. Use only columns that both files have.") from None
    try:
        X_all_my = np.asarray(X_all_my, dtype=float)
        X_test_my = np.asarray(X_test_my, dtype=float)
    except (TypeError, ValueError):
        raise AssertionError("build_my_features should return a matrix of numbers. "
                             "Turn a table into one with .to_numpy(dtype=float).") from None
    assert X_all_my.ndim == 2 and X_all_my.shape[0] == len(train) and X_test_my.shape[0] == len(test), \
        "build_my_features should return a matrix with one row for every row of the table it gets."
    assert X_all_my.shape[1] == X_test_my.shape[1], \
        f"build_my_features gives {X_all_my.shape[1]} columns for Train but {X_test_my.shape[1]} for Test. Use only columns that both files have."
    assert len(my_names) == X_all_my.shape[1], "names should have one entry for every column of X."
    assert np.all(np.isfinite(X_all_my)) and np.all(np.isfinite(X_test_my)), "Your features contain nan or inf."
    # the rows of Train in reverse order must give the same features in reverse order
    X_reversed = np.asarray(build_my_features(train[::-1].reset_index(drop=True))[0], dtype=float)
    assert np.allclose(X_reversed[::-1], X_all_my), \
        "Your features depend on the order of the rows. Row i of X should be the hour in row i of the table. If you sort the rows, put them back in their original order before you return X. A column of random numbers also changes with the order of the rows."
    assert np.allclose(count_from_target(target_from_count(y_all)), y_all), \
        "count_from_target(target_from_count(count)) should give the count back."
    assert isinstance(num_iters, (int, np.integer)) and num_iters >= 1, \
        "num_iters should be a whole number of 1 or more, for example 20000 and not 2e4."
    assert num_iters <= 100000, \
        "gradient_descent saves the cost of the first 100000 iterations only, so after that its prints stop changing. Use num_iters = 100000 or less."
    assert train.equals(train_loaded) and test.equals(test_loaded), \
        "build_my_features has changed train or test. Write new columns only into df, the copy that it gets. Then use Runtime > Run all."
    return X_all_my, X_test_my


X_all_my, X_test_my = build_and_check_my_features()
print(f"Your model has {X_all_my.shape[1]} features")
print("Check passed")

In [ ]:
# build your features, fit on days 1 to 14, then measure on the validation days
X_all_my, X_test_my = build_and_check_my_features()
w_my, b_my, mu_my, sigma_my, J_my = fit_linear_regression(X_all_my[~is_val], target_from_count(y_fit),
                                                          alpha, num_iters)
p_val_my = count_from_target(compute_model_output((X_all_my[is_val] - mu_my) / sigma_my, w_my, b_my))
p_val_my = np.clip(p_val_my, 0, None)
rmse_val_my = rmse(p_val_my, y_val)

n_saved = len(J_my)   # gradient_descent saves the cost of the first 100000 iterations only
last = min(max(n_saved // 10, 1), n_saved - 1)
print(f"\nCost after iteration {n_saved - 1 - last}: {J_my[-1 - last]:0.6g}. Cost after iteration {n_saved - 1}: {J_my[-1]:0.6g}")
print(f"Validation RMSE of your model:      {rmse_val_my:0.2f}")
print(f"Validation RMSE of the first model: {rmse_val_baseline:0.2f}")

# average true and predicted count in each hour of the validation days
val_rows = train[is_val].assign(predicted=p_val_my)

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for k, (wd, title) in enumerate([(1, "working days"), (0, "weekends and holidays")]):
    rows = val_rows[val_rows["workingday"] == wd]
    hourly_true = rows.groupby("hr")["count"].mean()
    hourly_pred = rows.groupby("hr")["predicted"].mean()
    ax[k].plot(hourly_true.index, hourly_true.values, marker="o", label="true count")
    ax[k].plot(hourly_pred.index, hourly_pred.values, marker="o", label="your model")
    ax[k].set_title(f"Validation days, {title}")
    ax[k].set_xlabel("hour of the day (hr)")
    ax[k].set_xticks(range(0, 24, 2))
ax[0].set_ylabel("average count")
ax[0].legend()
plt.show()

If the cost still goes down clearly at the end, gradient descent has not converged yet. Increase `num_iters` in the cell of your model, or check that your new columns are not almost copies of each other. Uncentred powers of the temperature are an example. With a log target the cost is computed on $\log(1 + \text{count})$, so it is much smaller than before. Compare models by their validation RMSE, not by their cost.

The plot compares the true count and your model's prediction, averaged over the validation days for each hour. It shows where your model is wrong. That helps you choose the next idea.

# 7 Retrain on all of Train and submit

When you have chosen your model, fit it again on all the rows of Train, days 1 to 19, as Section 4.4 did for the first model. Then predict the test hours and write the submission.

The cell below uses the cell of your model in Section 6 as you last ran it. If you have changed that cell since, run it again first. The cell below writes `my_submission.csv`, so `submission.csv` of Section 4.4 stays the file of the first model.

In [ ]:
# build your features and fit your model on all of Train, days 1 to 19
X_all_my, X_test_my = build_and_check_my_features()
w_my, b_my, mu_my, sigma_my, _ = fit_linear_regression(X_all_my, target_from_count(y_all), alpha, num_iters)

# predict the test hours with the mean and standard deviation of Train, and clip at 0
p_test_my = count_from_target(compute_model_output((X_test_my - mu_my) / sigma_my, w_my, b_my))
p_test_my = np.clip(p_test_my, 0, None)
write_submission(test["Id"], p_test_my, "my_submission.csv")

To submit, open the Files panel on the left of Colab, download `my_submission.csv` and upload it on the competition page on Kaggle. If you download it more than once, your browser may save the new copy as `my_submission (1).csv`. Upload the newest copy.

Kaggle lets you write a description for each submission. Write the validation RMSE of the model there. Before the deadline, select up to 2 submissions for the final ranking. Choose them by your validation RMSE, not by the public leaderboard.

# 8 Questions

Answer each question in a few sentences, in the cell below it.

### Question 2

Which change to the first model lowered the validation RMSE the most? How do you know that it was this change, and not chance?

Write your answer here.

### Question 3

Your validation RMSE and your score on the public leaderboard are not the same number. Give two reasons why they can differ.

Write your answer here.

### Question 4

Before clipping, the first model predicts a negative count for some hours. Why can a linear model predict a negative number of bikes? Explain why clipping the predictions at 0 can never make the RMSE worse.

Write your answer here.